In [1]:
import re
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr

from core import GRID_PARAMS
from geopy.distance import distance
import geopandas as gpd
from shapely.geometry import box

import echopype as ep

from core import TS_L_PARAMS
from grid import create_boundary_gdf, create_grid_from_bounds

In [2]:
data_path = Path("/Users/wujung/code_git/echodataflow/temp_bio")
csv_path = data_path / "bio_csv"
NASC_path = data_path / "nasc_zarr"

# Initialize haul info dataframe
df_haul_info = pd.DataFrame(
    columns=["operation_number", "timestamp", "latitude", "longitude"]
)
df_haul_info.to_csv(data_path / "haul_info.csv", index=False)

In [6]:
from flows_biology import (
    get_valid_hauls,
    get_count_from_length_specimen,
    add_stratum,
    get_length_weight_regression,
    get_sigma_bs_mean_stratum,
    get_weight_mean_stratum,
)

In [13]:
from core import TS_L_PARAMS, GRID_PARAMS, INFO_DATAFRAME_MAPPING

In [5]:
bio_path: str = csv_path
date_prefix: str = "202407"
species_code: int = 22500
haul_info_filename: str = data_path / "haul_info.csv"

In [6]:
# Get all filenames
bio_filenames = {
    "length": list(bio_path.glob(f"{date_prefix}_{species_code}_*_lf.csv")),
    "specimen": list(bio_path.glob(f"{date_prefix}_{species_code}_*_spec.csv")),
    "catch": list(bio_path.glob(f"{date_prefix}_*_catch_perc.csv")),
    "info": list(bio_path.glob(f"{date_prefix}_*_operation_info.csv")),
}

# Get valid haul numbers (those with 4 files)
hauls_valid = get_valid_hauls(date_prefix, species_code, bio_filenames)

# Get hauls to process
df_haul_info = pd.read_csv(haul_info_filename)
hauls_processed = set(df_haul_info["operation_number"].unique())
hauls_to_process = list(hauls_valid.difference(hauls_processed))


In [7]:
hauls_to_process

[3,
 8,
 11,
 12,
 15,
 17,
 20,
 21,
 22,
 23,
 24,
 25,
 26,
 27,
 28,
 31,
 32,
 33,
 35,
 36,
 38,
 40,
 41,
 43,
 44,
 47,
 49,
 50,
 52,
 54,
 55,
 59,
 60,
 61,
 62,
 99]

In [8]:
haul_num_all = [3, 25, 60]
# Consolidate length count for each haul

df_length = []
df_specimen = []
df_info = []
for haul_num in haul_num_all:
    df_length.append(
        pd.read_csv(
            csv_path / f"{date_prefix}_{species_code}_{haul_num:03d}_lf.csv", index_col=0
        ).reset_index()
    )
    df_specimen.append(
        pd.read_csv(
            csv_path / f"{date_prefix}_{species_code}_{haul_num:03d}_spec.csv", index_col=0
        ).reset_index()
    )
    df_info.append(
        pd.read_csv(
            csv_path / f"{date_prefix}_{haul_num:03d}_operation_info.csv", index_col=0
        ).rename(columns=INFO_DATAFRAME_MAPPING).reset_index()  # reset index to get operation_number into a column
    )
df_length = pd.concat(df_length, ignore_index=True)
df_specimen = pd.concat(df_specimen, ignore_index=True)
df_info = pd.concat(df_info, ignore_index=True)

In [9]:
df_length

,operation_number,partition,species,sex,rounded_length,frequency
0,3,Codend,Pacific hake - Merluccius productus,Female,21.0,1
1,3,Codend,Pacific hake - Merluccius productus,Female,22.0,12
2,3,Codend,Pacific hake - Merluccius productus,Female,23.0,26
3,3,Codend,Pacific hake - Merluccius productus,Female,24.0,82
4,3,Codend,Pacific hake - Merluccius productus,Female,25.0,89
...,...,...,...,...,...,...
89,60,Codend,Pacific hake - Merluccius productus,Male,47.0,12
90,60,Codend,Pacific hake - Merluccius productus,Male,48.0,11
91,60,Codend,Pacific hake - Merluccius productus,Male,49.0,3
92,60,Codend,Pacific hake - Merluccius productus,Male,50.0,2


In [10]:
df_specimen["rounded_length"] = df_specimen["length"].round(0)

In [11]:
df_specimen

,operation_number,partition,species,length_type,length,sex,organism_weight,barcode,rounded_length
0,3,Codend,Pacific hake - Merluccius productus,fork_length,20.7,Male,0.070,100634014,21.0
1,3,Codend,Pacific hake - Merluccius productus,fork_length,20.8,Male,0.056,100634181,21.0
2,3,Codend,Pacific hake - Merluccius productus,fork_length,21.4,Male,0.055,100634031,21.0
3,3,Codend,Pacific hake - Merluccius productus,fork_length,21.6,Female,0.065,100634042,22.0
4,3,Codend,Pacific hake - Merluccius productus,fork_length,21.9,Male,0.075,100634013,22.0
...,...,...,...,...,...,...,...,...,...
185,60,Codend,Pacific hake - Merluccius productus,fork_length,46.9,Female,0.610,100640908,47.0
186,60,Codend,Pacific hake - Merluccius productus,fork_length,47.8,Male,0.670,100640913,48.0
187,60,Codend,Pacific hake - Merluccius productus,fork_length,49.3,Female,0.615,100640907,49.0
188,60,Codend,Pacific hake - Merluccius productus,fork_length,49.4,Male,0.805,100640912,49.0


In [12]:
df_length_count = get_count_from_length_specimen(df_length, df_specimen)

In [13]:
df_length_count

,sex,rounded_length,frequency,operation_number
0,Female,21.0,1,3
1,Female,22.0,15,3
2,Female,23.0,33,3
3,Female,24.0,96,3
4,Female,25.0,103,3
...,...,...,...,...
89,Male,48.0,12,60
90,Male,49.0,4,60
91,Male,50.0,2,60
92,Male,51.0,1,25


In [14]:
df_specimen = pd.merge(
    df_specimen,
    df_info,
    on="operation_number",
    how="left"
)
df_length_count = pd.merge(
    df_length_count,
    df_info,
    on="operation_number",
    how="left"
)

In [15]:
df_specimen

,operation_number,partition,species,length_type,length,sex,organism_weight,barcode,rounded_length,timestamp,latitude,longitude
0,3,Codend,Pacific hake - Merluccius productus,fork_length,20.7,Male,0.070,100634014,21.0,20240709002246,37.155009,-122.754236
1,3,Codend,Pacific hake - Merluccius productus,fork_length,20.8,Male,0.056,100634181,21.0,20240709002246,37.155009,-122.754236
2,3,Codend,Pacific hake - Merluccius productus,fork_length,21.4,Male,0.055,100634031,21.0,20240709002246,37.155009,-122.754236
3,3,Codend,Pacific hake - Merluccius productus,fork_length,21.6,Female,0.065,100634042,22.0,20240709002246,37.155009,-122.754236
4,3,Codend,Pacific hake - Merluccius productus,fork_length,21.9,Male,0.075,100634013,22.0,20240709002246,37.155009,-122.754236
...,...,...,...,...,...,...,...,...,...,...,...,...
185,60,Codend,Pacific hake - Merluccius productus,fork_length,46.9,Female,0.610,100640908,47.0,20240817230607,44.118454,-124.979059
186,60,Codend,Pacific hake - Merluccius productus,fork_length,47.8,Male,0.670,100640913,48.0,20240817230607,44.118454,-124.979059
187,60,Codend,Pacific hake - Merluccius productus,fork_length,49.3,Female,0.615,100640907,49.0,20240817230607,44.118454,-124.979059
188,60,Codend,Pacific hake - Merluccius productus,fork_length,49.4,Male,0.805,100640912,49.0,20240817230607,44.118454,-124.979059


In [16]:
# Add stratrum info to df_specimen and df_length_count based on latitude
df_stratum = pd.read_csv(data_path / "inpfc_def.csv", index_col=0).reset_index().rename(
    columns={"stratum_num": "stratum"}
)
df_specimen = add_stratum(df_specimen, df_stratum)
df_length_count = add_stratum(df_length_count, df_stratum)

In [17]:
df_length_weight_regression = get_length_weight_regression(df_specimen)

In [18]:
df_length_weight_regression

,index,sex,stratum,p1,p2
0,0,female,2,2.900909,-5.063268
1,1,female,3,2.626181,-4.595349
2,2,female,4,2.438220,-4.275218
3,3,male,2,2.726465,-4.821799
4,4,male,3,2.641715,-4.609706
5,5,male,4,2.679331,-4.656770
6,0,all,2,2.802601,-4.926800
7,1,all,3,2.615129,-4.572457
8,2,all,4,2.525222,-4.411694


In [19]:
df_stratum = pd.merge(
    df_stratum,
    get_sigma_bs_mean_stratum(df_length_count),
    on="stratum",
    how="outer"
)
df_stratum = pd.merge(
    df_stratum,
    get_weight_mean_stratum(df_length_count, df_length_weight_regression),
    on="stratum",
    how="outer"
)

In [20]:
df_stratum

,stratum,latitude_northern_limit,stratum_name,sigma_bs_mean,weight_mean
0,1,36.0000,Conception,NaN,NaN
1,2,40.5000,Monterey,0.000100,0.099443
2,3,43.0000,Eureka,0.000229,0.365021
3,4,45.7667,South Columbia,0.000298,0.530063
4,5,48.5000,North Columbia - US/CAN border,NaN,NaN
5,6,55.0000,Canada,NaN,NaN


In [21]:
df_haul_info

,operation_number,timestamp,latitude,longitude


In [22]:
# Initialize dataframes
df_haul_info_all = pd.DataFrame(
    columns=["operation_number", "timestamp", "latitude", "longitude"]
)
df_haul_info_all.to_csv(data_path / "haul_info_all.csv")

df_specimen_all = pd.DataFrame(
    columns=["operation_number", "partition", "species", "sex", "rounded_length", "frequency"]
)
df_specimen_all.to_csv(data_path / "specimen_all.csv")

df_length_all = pd.DataFrame(
    columns=[
        "operation_number", "partition", "species",
        "length_type", "length", "sex", "organism_weight", "barcode"
    ]
)
df_length_all.to_csv(data_path / "length_all.csv")

df_length_count_all = pd.DataFrame(
    columns=["sex", "rounded_length", "frequency", "operation_number"]
)
df_length_count_all.to_csv(data_path / "length_count_all.csv")

In [27]:
df_specimen_all = pd.concat([df_specimen_all, df_specimen], ignore_index=True)
df_length_count_all = pd.concat([df_length_count, df_length_count], ignore_index=True)
df_haul_info_all = pd.concat([df_haul_info_all, df_info], ignore_index=True)

In [28]:
df_length_count_all

,sex,rounded_length,frequency,operation_number,timestamp,latitude,longitude,stratum,sigma_bs
0,Female,21.0,1,3,20240709002246,37.155009,-122.754236,2,0.000070
1,Female,22.0,15,3,20240709002246,37.155009,-122.754236,2,0.000077
2,Female,23.0,33,3,20240709002246,37.155009,-122.754236,2,0.000084
3,Female,24.0,96,3,20240709002246,37.155009,-122.754236,2,0.000091
4,Female,25.0,103,3,20240709002246,37.155009,-122.754236,2,0.000099
...,...,...,...,...,...,...,...,...,...
183,Male,48.0,12,60,20240817230607,44.118454,-124.979059,4,0.000365
184,Male,49.0,4,60,20240817230607,44.118454,-124.979059,4,0.000381
185,Male,50.0,2,60,20240817230607,44.118454,-124.979059,4,0.000396
186,Male,51.0,1,25,20240719182733,40.640122,-124.595902,3,0.000412


In [3]:
data_path = Path("/Users/wujung/code_git/echodataflow/temp_bio")
csv_path = data_path / "bio_csv"
NASC_path = data_path / "nasc_zarr"

# Initialize dataframes
df_haul_info_all = pd.DataFrame(
    columns=["operation_number", "timestamp", "latitude", "longitude"]
)
df_specimen_all = pd.DataFrame(
    columns=["operation_number", "partition", "species", "sex", "rounded_length", "frequency"]
)
df_length_all = pd.DataFrame(
    columns=[
        "operation_number", "partition", "species",
        "length_type", "length", "sex", "organism_weight", "barcode"
    ]
)
df_length_count_all = pd.DataFrame(
    columns=["sex", "rounded_length", "frequency", "operation_number"]
)
df_haul_info_all.to_csv(data_path / "haul_info_all.csv")
df_specimen_all.to_csv(data_path / "specimen_all.csv")
df_length_all.to_csv(data_path / "length_all.csv")
df_length_count_all.to_csv(data_path / "length_count_all.csv")


In [30]:
bio_path: str = csv_path
date_prefix: str = "202407"
species_code: int = 22500
haul_info_all_path: str = data_path / "haul_info_all.csv"
specimen_all_path: str = data_path / "specimen_all.csv"
length_all_path: str = data_path / "length_all.csv"
length_count_all_path: str = data_path / "length_count_all.csv"
stratum_mean_path: str = data_path / "stratum_mean.csv"

In [10]:
# Get all filenames
bio_filenames = {
    "length": list(bio_path.glob(f"{date_prefix}_{species_code}_*_lf.csv")),
    "specimen": list(bio_path.glob(f"{date_prefix}_{species_code}_*_spec.csv")),
    "catch": list(bio_path.glob(f"{date_prefix}_*_catch_perc.csv")),
    "info": list(bio_path.glob(f"{date_prefix}_*_operation_info.csv")),
}

# Get valid haul numbers (those with 4 files)
hauls_valid = get_valid_hauls(date_prefix, species_code, bio_filenames)

# Get hauls to process
df_haul_info_all = pd.read_csv(haul_info_path, index_col=0)
if df_haul_info_all.empty:
    hauls_processed = set()
else:
    hauls_processed = set(df_haul_info_all["operation_number"].unique())
hauls_to_process = list(hauls_valid.difference(hauls_processed))

if not hauls_to_process:  # if there are hauls to process
    print(f"No hauls to process for {date_prefix} with species code {species_code}.")

In [14]:
if not hauls_to_process:  # if there are hauls to process
    print(f"No hauls to process for {date_prefix} with species code {species_code}.")
    # return
else:
    # Load dataframes from all hauls to process
    hauls_to_process = hauls_to_process[:3]
    df_length = []
    df_specimen = []
    df_info = []
    for haul_num in hauls_to_process:
        df_length.append(
            pd.read_csv(
                csv_path / f"{date_prefix}_{species_code}_{haul_num:03d}_lf.csv", index_col=0
            ).reset_index()
        )
        df_specimen.append(
            pd.read_csv(
                csv_path / f"{date_prefix}_{species_code}_{haul_num:03d}_spec.csv", index_col=0
            ).reset_index()
        )
        df_info.append(
            pd.read_csv(
                csv_path / f"{date_prefix}_{haul_num:03d}_operation_info.csv", index_col=0
            ).rename(columns=INFO_DATAFRAME_MAPPING).reset_index()  # reset index to get operation_number into a column
        )
    df_length = pd.concat(df_length, ignore_index=True)
    df_specimen = pd.concat(df_specimen, ignore_index=True)
    df_info = pd.concat(df_info, ignore_index=True)

    # Combined length frequency from length and specimen dataframes
    df_length_count = get_count_from_length_specimen(df_length, df_specimen)

    # Add haul number and lat/lon for downstream stratification
    df_specimen = pd.merge(
        df_specimen,
        df_info,
        on="operation_number",
        how="left"
    )
    df_length_count = pd.merge(
        df_length_count,
        df_info,
        on="operation_number",
        how="left"
    )

    # Update df_haul_info_all, df_specimen_all, df_length_all
    df_specimen_all = pd.read_csv(specimen_all_path, index_col=0)
    df_length_all = pd.read_csv(length_all_path, index_col=0)
    df_length_count_all = pd.read_csv(length_count_all_path, index_col=0)
    df_haul_info_all = pd.read_csv(haul_info_path, index_col=0)

    df_specimen_all = pd.concat([df_specimen_all, df_specimen], ignore_index=True)
    df_length_all = pd.concat([df_length_all, df_length], ignore_index=True)
    df_length_count_all = pd.concat([df_length_count_all, df_length_count], ignore_index=True)
    df_haul_info_all = pd.concat([df_haul_info_all, df_info], ignore_index=True)

    df_specimen_all.to_csv(specimen_all_path)
    df_length_all.to_csv(length_all_path)
    df_length_count_all.to_csv(length_count_all_path)
    df_haul_info_all.to_csv(haul_info_path)

    # Add stratrum info to df_specimen and df_length_count based on latitude
    df_stratum = pd.read_csv(data_path / "inpfc_def.csv", index_col=0).reset_index().rename(
        columns={"stratum_num": "stratum"}
    )
    df_specimen_all = add_stratum(df_specimen_all, df_stratum)
    df_length_count_all = add_stratum(df_length_count_all, df_stratum)

    # Compute length-weight relationship for each stratum
    # Separately for: male, female, all fish combined
    df_length_weight_regression = get_length_weight_regression(df_specimen_all)

    # Compute mean sigma_bs and mean weight for each stratum
    # columns: stratum, sigma_bs_mean, weight_mean
    df_stratum = pd.merge(
        df_stratum,
        get_sigma_bs_mean_stratum(df_length_count_all),
        on="stratum",
        how="outer"
    )
    df_stratum = pd.merge(
        df_stratum,
        get_weight_mean_stratum(df_length_count_all, df_length_weight_regression),
        on="stratum",
        how="outer"
    )
    df_stratum.to_csv(stratum_mean_path)


/var/folders/1m/8nxc8r_900778tkhqfgh0nqh0000gn/T/ipykernel_82034/880271559.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_specimen_all = pd.concat([df_specimen_all, df_specimen], ignore_index=True)
/var/folders/1m/8nxc8r_900778tkhqfgh0nqh0000gn/T/ipykernel_82034/880271559.py:55: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_length_count_all = pd.concat([df_length_count_all, df_length_count], ignore_index=True)
/var/folders/1m/8nxc8r_900778tkhqfgh0nqh0000gn/T/ipykernel_82034/880271559

In [24]:
# Get all filenames
bio_filenames = {
    "length": list(bio_path.glob(f"{date_prefix}_{species_code}_*_lf.csv")),
    "specimen": list(bio_path.glob(f"{date_prefix}_{species_code}_*_spec.csv")),
    "catch": list(bio_path.glob(f"{date_prefix}_*_catch_perc.csv")),
    "info": list(bio_path.glob(f"{date_prefix}_*_operation_info.csv")),
}

# Get valid haul numbers (those with 4 files)
hauls_valid = get_valid_hauls(date_prefix, species_code, bio_filenames)

# Get hauls to process
df_haul_info_all = pd.read_csv(haul_info_path, index_col=0)
if df_haul_info_all.empty:
    hauls_processed = set()
else:
    hauls_processed = set(df_haul_info_all["operation_number"].unique())
hauls_to_process = list(hauls_valid.difference(hauls_processed))

In [25]:
hauls_processed

{3.0, 8.0, 11.0}

In [26]:
len(hauls_to_process)

33

In [28]:
hauls_to_process = hauls_to_process[20:24]
hauls_to_process

[43, 44, 47, 49]

In [29]:
if not hauls_to_process:  # if there are hauls to process
    print(f"No hauls to process for {date_prefix} with species code {species_code}.")
    # return
else:
    # Load dataframes from all hauls to process
    df_length = []
    df_specimen = []
    df_info = []
    for haul_num in hauls_to_process:
        df_length.append(
            pd.read_csv(
                csv_path / f"{date_prefix}_{species_code}_{haul_num:03d}_lf.csv", index_col=0
            ).reset_index()
        )
        df_specimen.append(
            pd.read_csv(
                csv_path / f"{date_prefix}_{species_code}_{haul_num:03d}_spec.csv", index_col=0
            ).reset_index()
        )
        df_info.append(
            pd.read_csv(
                csv_path / f"{date_prefix}_{haul_num:03d}_operation_info.csv", index_col=0
            ).rename(columns=INFO_DATAFRAME_MAPPING).reset_index()  # reset index to get operation_number into a column
        )
    df_length = pd.concat(df_length, ignore_index=True)
    df_specimen = pd.concat(df_specimen, ignore_index=True)
    df_info = pd.concat(df_info, ignore_index=True)

    # Combined length frequency from length and specimen dataframes
    df_length_count = get_count_from_length_specimen(df_length, df_specimen)

    # Add haul number and lat/lon for downstream stratification
    df_specimen = pd.merge(
        df_specimen,
        df_info,
        on="operation_number",
        how="left"
    )
    df_length_count = pd.merge(
        df_length_count,
        df_info,
        on="operation_number",
        how="left"
    )

    # Update df_haul_info_all, df_specimen_all, df_length_all
    df_specimen_all = pd.read_csv(specimen_all_path, index_col=0)
    df_length_all = pd.read_csv(length_all_path, index_col=0)
    df_length_count_all = pd.read_csv(length_count_all_path, index_col=0)
    df_haul_info_all = pd.read_csv(haul_info_path, index_col=0)

    df_specimen_all = pd.concat([df_specimen_all, df_specimen], ignore_index=True)
    df_length_all = pd.concat([df_length_all, df_length], ignore_index=True)
    df_length_count_all = pd.concat([df_length_count_all, df_length_count], ignore_index=True)
    df_haul_info_all = pd.concat([df_haul_info_all, df_info], ignore_index=True)

    df_specimen_all.to_csv(specimen_all_path)
    df_length_all.to_csv(length_all_path)
    df_length_count_all.to_csv(length_count_all_path)
    df_haul_info_all.to_csv(haul_info_path)

    # Add stratrum info to df_specimen and df_length_count based on latitude
    df_stratum = pd.read_csv(data_path / "inpfc_def.csv", index_col=0).reset_index().rename(
        columns={"stratum_num": "stratum"}
    )
    df_specimen_all = add_stratum(df_specimen_all, df_stratum)
    df_length_count_all = add_stratum(df_length_count_all, df_stratum)

    # Compute length-weight relationship for each stratum
    # Separately for: male, female, all fish combined
    df_length_weight_regression = get_length_weight_regression(df_specimen_all)

    # Compute mean sigma_bs and mean weight for each stratum
    # columns: stratum, sigma_bs_mean, weight_mean
    df_stratum = pd.merge(
        df_stratum,
        get_sigma_bs_mean_stratum(df_length_count_all),
        on="stratum",
        how="outer"
    )
    df_stratum = pd.merge(
        df_stratum,
        get_weight_mean_stratum(df_length_count_all, df_length_weight_regression),
        on="stratum",
        how="outer"
    )
    df_stratum.to_csv(stratum_mean_path)

In [31]:
# Get all filenames
bio_filenames = {
    "length": list(bio_path.glob(f"{date_prefix}_{species_code}_*_lf.csv")),
    "specimen": list(bio_path.glob(f"{date_prefix}_{species_code}_*_spec.csv")),
    "catch": list(bio_path.glob(f"{date_prefix}_*_catch_perc.csv")),
    "info": list(bio_path.glob(f"{date_prefix}_*_operation_info.csv")),
}

# Get valid haul numbers (those with 4 files)
hauls_valid = get_valid_hauls(date_prefix, species_code, bio_filenames)

# Get hauls to process
df_haul_info_all = pd.read_csv(haul_info_all_path, index_col=0)
if df_haul_info_all.empty:
    hauls_processed = set()
else:
    hauls_processed = set(df_haul_info_all["operation_number"].unique())
hauls_to_process = list(hauls_valid.difference(hauls_processed))

In [34]:
hauls_to_process = hauls_to_process[-5:]

In [35]:
if not hauls_to_process:  # if there are hauls to process
    print(f"No hauls to process for {date_prefix} with species code {species_code}.")
    # return
else:
    # Load dataframes from all hauls to process
    df_length = []
    df_specimen = []
    df_info = []
    for haul_num in hauls_to_process:
        df_length.append(
            pd.read_csv(
                csv_path / f"{date_prefix}_{species_code}_{haul_num:03d}_lf.csv", index_col=0
            ).reset_index()
        )
        df_specimen.append(
            pd.read_csv(
                csv_path / f"{date_prefix}_{species_code}_{haul_num:03d}_spec.csv", index_col=0
            ).reset_index()
        )
        df_info.append(
            pd.read_csv(
                csv_path / f"{date_prefix}_{haul_num:03d}_operation_info.csv", index_col=0
            ).rename(columns=INFO_DATAFRAME_MAPPING).reset_index()  # reset index to get operation_number into a column
        )
    df_length = pd.concat(df_length, ignore_index=True)
    df_specimen = pd.concat(df_specimen, ignore_index=True)
    df_info = pd.concat(df_info, ignore_index=True)

    # Combined length frequency from length and specimen dataframes
    df_length_count = get_count_from_length_specimen(df_length, df_specimen)

    # Add haul number and lat/lon for downstream stratification
    df_specimen = pd.merge(
        df_specimen,
        df_info,
        on="operation_number",
        how="left"
    )
    df_length_count = pd.merge(
        df_length_count,
        df_info,
        on="operation_number",
        how="left"
    )

    # Update df_haul_info_all, df_specimen_all, df_length_all
    df_specimen_all = pd.read_csv(specimen_all_path, index_col=0)
    df_length_all = pd.read_csv(length_all_path, index_col=0)
    df_length_count_all = pd.read_csv(length_count_all_path, index_col=0)
    df_haul_info_all = pd.read_csv(haul_info_all_path, index_col=0)

    df_specimen_all = pd.concat([df_specimen_all, df_specimen], ignore_index=True)
    df_length_all = pd.concat([df_length_all, df_length], ignore_index=True)
    df_length_count_all = pd.concat([df_length_count_all, df_length_count], ignore_index=True)
    df_haul_info_all = pd.concat([df_haul_info_all, df_info], ignore_index=True)

    df_specimen_all.to_csv(specimen_all_path)
    df_length_all.to_csv(length_all_path)
    df_length_count_all.to_csv(length_count_all_path)
    df_haul_info_all.to_csv(haul_info_all_path)

    # Add stratrum info to df_specimen and df_length_count based on latitude
    df_stratum = pd.read_csv(data_path / "inpfc_def.csv", index_col=0).reset_index().rename(
        columns={"stratum_num": "stratum"}
    )
    df_specimen_all = add_stratum(df_specimen_all, df_stratum)
    df_length_count_all = add_stratum(df_length_count_all, df_stratum)

    # Compute length-weight relationship for each stratum
    # Separately for: male, female, all fish combined
    df_length_weight_regression = get_length_weight_regression(df_specimen_all)

    # Compute mean sigma_bs and mean weight for each stratum
    # columns: stratum, sigma_bs_mean, weight_mean
    df_stratum = pd.merge(
        df_stratum,
        get_sigma_bs_mean_stratum(df_length_count_all),
        on="stratum",
        how="outer"
    )
    df_stratum = pd.merge(
        df_stratum,
        get_weight_mean_stratum(df_length_count_all, df_length_weight_regression),
        on="stratum",
        how="outer"
    )
    df_stratum.to_csv(stratum_mean_path)

In [37]:
# Path to dataframe containing processed NASC filenames
NASC_processed_path = data_path / "nasc_processed.csv"
if not NASC_processed_path.exists():
    # Create an empty dataframe to store processed NASC filenames
    df_NASC_processed = pd.DataFrame(columns=["filename"])
    df_NASC_processed.to_csv(NASC_processed_path, index=False)


In [42]:
from flows_biology import (
    combine_NASC_dataset_to_dataframe,
    griddify_NASC,
    update_grid
)

In [50]:
NASC_filenames_all = list(NASC_path.glob("*.zarr"))
NASC_filenames_all = [f.name for f in NASC_filenames_all]

In [67]:
# Path to dataframe containing processed NASC filenames
NASC_processed_path = data_path / "nasc_processed.csv"
if not NASC_processed_path.exists():
    # Create an empty dataframe to store processed NASC filenames
    df_NASC_processed = pd.DataFrame(columns="filename")
    df_NASC_processed.to_csv(NASC_processed_path, index=False)


In [74]:
df_NASC_processed = pd.DataFrame(
    data=NASC_filenames_all[:3], columns=["filename"]
)
df_NASC_processed.to_csv(NASC_processed_path, index=False)

In [78]:
num_NASC_ignore = 3

In [88]:
NASC_all = list(NASC_path.glob("*.zarr"))
NASC_all = sorted([f.name for f in NASC_all])
NASC_all = set(NASC_all[:-num_NASC_ignore])
NASC_processed = set(
    pd.read_csv(NASC_processed_path, index_col=0).reset_index()["filename"]
)

In [91]:
NASC_to_process = NASC_all.difference(NASC_processed)

In [99]:
def combine_NASC_dataset_to_dataframe(
    NASC_path: Path,
    NASC_filenames: list
) -> pd.DataFrame:
    """
    Combine multiple NASC datasets into a single DataFrame.
    """
    df_NASC_list = []
    for nascf in NASC_filenames:
        ds_NASC = xr.open_zarr(NASC_path / nascf)
        ds_NASC = ep.consolidate.swap_dims_channel_frequency(ds_NASC)
        df_NASC = ds_NASC.sum("depth").sel(frequency_nominal=38000).to_dataframe()
        df_NASC["filename"] = nascf
        df_NASC_list.append(df_NASC)
    return pd.concat(df_NASC_list, ignore_index=True)

In [100]:
df_NASC = combine_NASC_dataset_to_dataframe(NASC_path, NASC_to_process)

In [101]:
df_NASC

,NASC,channel,frequency_nominal,latitude,longitude,ping_time,filename
0,0.0,WBT 400143-15 ES38B_ES,38000.0,37.466030,-122.985679,2024-07-08 15:33:40.000,win_1720452580_1720455075_NASC.zarr
1,0.0,WBT 400143-15 ES38B_ES,38000.0,37.461271,-122.975942,2024-07-08 15:40:25.000,win_1720452580_1720455075_NASC.zarr
2,0.0,WBT 400143-15 ES38B_ES,38000.0,37.455019,-122.969136,2024-07-08 15:45:42.500,win_1720452580_1720455075_NASC.zarr
3,0.0,WBT 400143-15 ES38B_ES,38000.0,37.448527,-122.962649,2024-07-08 15:50:57.500,win_1720452580_1720455075_NASC.zarr
4,0.0,WBT 400143-15 ES38B_ES,38000.0,37.442500,-122.955560,2024-07-08 15:56:22.500,win_1720452580_1720455075_NASC.zarr
...,...,...,...,...,...,...,...
3272,0.0,WBT 400143-15 ES38B_ES,38000.0,38.291722,-124.173648,2024-07-08 00:42:07.500,win_1720397470_1720399965_NASC.zarr
3273,0.0,WBT 400143-15 ES38B_ES,38000.0,38.284591,-124.168143,2024-07-08 00:45:05.000,win_1720397470_1720399965_NASC.zarr
3274,0.0,WBT 400143-15 ES38B_ES,38000.0,38.277620,-124.162622,2024-07-08 00:48:00.000,win_1720397470_1720399965_NASC.zarr
3275,0.0,WBT 400143-15 ES38B_ES,38000.0,38.270564,-124.157037,2024-07-08 00:50:57.500,win_1720397470_1720399965_NASC.zarr


In [104]:
NASC_all_path: str = data_path / "NASC_all.csv"
NASC_all_path

PosixPath('/Users/wujung/code_git/echodataflow/temp_bio/NASC_all.csv')

In [105]:
if not NASC_all_path.exists():
    df_NASC_all = pd.DataFrame()

In [107]:
pd.concat([df_NASC_all, df_NASC], ignore_index=True)

,NASC,channel,frequency_nominal,latitude,longitude,ping_time,filename
0,0.0,WBT 400143-15 ES38B_ES,38000.0,37.466030,-122.985679,2024-07-08 15:33:40.000,win_1720452580_1720455075_NASC.zarr
1,0.0,WBT 400143-15 ES38B_ES,38000.0,37.461271,-122.975942,2024-07-08 15:40:25.000,win_1720452580_1720455075_NASC.zarr
2,0.0,WBT 400143-15 ES38B_ES,38000.0,37.455019,-122.969136,2024-07-08 15:45:42.500,win_1720452580_1720455075_NASC.zarr
3,0.0,WBT 400143-15 ES38B_ES,38000.0,37.448527,-122.962649,2024-07-08 15:50:57.500,win_1720452580_1720455075_NASC.zarr
4,0.0,WBT 400143-15 ES38B_ES,38000.0,37.442500,-122.955560,2024-07-08 15:56:22.500,win_1720452580_1720455075_NASC.zarr
...,...,...,...,...,...,...,...
3272,0.0,WBT 400143-15 ES38B_ES,38000.0,38.291722,-124.173648,2024-07-08 00:42:07.500,win_1720397470_1720399965_NASC.zarr
3273,0.0,WBT 400143-15 ES38B_ES,38000.0,38.284591,-124.168143,2024-07-08 00:45:05.000,win_1720397470_1720399965_NASC.zarr
3274,0.0,WBT 400143-15 ES38B_ES,38000.0,38.277620,-124.162622,2024-07-08 00:48:00.000,win_1720397470_1720399965_NASC.zarr
3275,0.0,WBT 400143-15 ES38B_ES,38000.0,38.270564,-124.157037,2024-07-08 00:50:57.500,win_1720397470_1720399965_NASC.zarr
